# 3 · Schema matching  (per article)

For each **distinct OIE type** an article used, retrieve the nearest schema.org classes by
embedding similarity; every entity inherits its type's match. Matching is done at the type level
(the mention is deliberately not embedded), so the same type always resolves the same way.

Two document pools are compared, holding the query side fixed:
- **baseline** — the terse schema.org comments (`schema_classes.json`)
- **gemini**   — the Gemini-enriched definitions (`schema_classes_enriched.json`, built in nb 1)

and two embedding models (bge-small, e5-base). Set `ARTICLE_ID`, or run the final **batch** cell.
Canonical per-model outputs are written to
`results/<ARTICLE_ID>/schema_matching_<model>{,_by_type}.json` (from the **gemini** pool).

In [14]:
# --- Config -------------------------------------------------------------------------------
import json, re
from pathlib import Path
import numpy as np

ARTICLE_ID  = "art2"                     # which article to match (or use the batch cell below)
SCHEMA_DIR  = Path("schema_org_files")   # shared pools (built by 1_schema_enrichment.ipynb)
RESULTS_DIR = Path("results")            # reads oie_entities.json, writes schema_matching_*.json

TOP_K = 5
ALIGN_QUERY_STYLE = True                 # append "Also called <synonyms>." to each OIE type query
                                         # (mirrors the enriched doc format; no-op without synonyms)

# Embedding models to compare. Each family needs its OWN prefix convention, applied separately to
# queries and documents. bge: query gets an instruction, documents get NO prefix. e5: query gets
# "query: ", documents get "passage: " (REQUIRED - e5 was trained this way; dropping the passage
# prefix badly hurts its scores).
EMBED_MODELS = ["BAAI/bge-small-en-v1.5", "intfloat/e5-base-v2"]

def embed_prefixes(model_name):
    n = model_name.lower()
    if "e5" in n:  return "query: ", "passage: "
    if "bge" in n: return "Represent this sentence for searching relevant passages: ", ""
    return "", ""   # gte / generic: no special prefixes

print("article:", ARTICLE_ID, "| embed:", EMBED_MODELS)

article: art2 | embed: ['BAAI/bge-small-en-v1.5', 'intfloat/e5-base-v2']


In [2]:
# Run once to install dependencies (sentence-transformers pulls in torch, a large first download).
%pip install -q sentence-transformers numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Load the document pools + define retrieval

In [3]:
# Document pools are shared across articles (built once by 1_schema_enrichment.ipynb).
# baseline = terse schema.org comments; gemini = Gemini-enriched definitions. Both use the same
# "Label (path): comment" embedding_text, so any difference is purely the enrichment.
baseline_pool = json.loads((SCHEMA_DIR / "schema_classes.json").read_text())
gemini_pool   = json.loads((SCHEMA_DIR / "schema_classes_enriched.json").read_text())
POOLS = {"baseline": baseline_pool, "gemini": gemini_pool}
print(f"{len(baseline_pool)} baseline classes | "
      f"{sum(c.get('enriched') for c in gemini_pool)}/{len(gemini_pool)} enriched")

from sentence_transformers import SentenceTransformer
_MODELS = {}   # load each embedding model once, reuse across articles
def get_model(name):
    if name not in _MODELS:
        _MODELS[name] = SentenceTransformer(name)
    return _MODELS[name]

def load_queries(article_id):
    """One query per DISTINCT OIE type = 'type: definition' (mention excluded), optionally with
    'Also called <synonyms>.' appended. Returns (oie, entities, oie_types, type_texts)."""
    oie = json.loads((RESULTS_DIR / article_id / "oie_entities.json").read_text())
    type_info = {t["type"]: t for t in oie.get("types", [])}
    entities  = oie["entities"]
    oie_types = list(dict.fromkeys(e["type"] for e in entities))
    def type_query_text(t):
        info = type_info.get(t, {})
        q = f"{t}: {info.get('definition', '')}".strip()
        if ALIGN_QUERY_STYLE and info.get("synonyms"):
            q = f"{q} Also called {', '.join(info['synonyms'])}."
        return q
    return oie, entities, oie_types, [type_query_text(t) for t in oie_types]

def retrieve(model, pool, type_texts, oie_types, qp, dp):
    """Embed queries + a document pool with the model's OWN prefixes; top-k per OIE type."""
    q_emb = np.asarray(model.encode([qp + t for t in type_texts],
                                    normalize_embeddings=True, batch_size=64))
    d_emb = np.asarray(model.encode([dp + c["embedding_text"] for c in pool],
                                    normalize_embeddings=True, batch_size=64, show_progress_bar=True))
    sims = q_emb @ d_emb.T
    topk = np.argsort(-sims, axis=1)[:, :TOP_K]
    return {t: [{"label": pool[j]["label"], "score": round(float(sims[ti, j]), 3),
                 "path": pool[j].get("path", ""),
                 "comment": pool[j].get("enriched_comment") or pool[j].get("comment", "")}
                for j in topk[ti]]
            for ti, t in enumerate(oie_types)}

def match_article(article_id):
    """Retrieve top-k for every (embedding model x pool). Returns (oie, entities, oie_types, results)
    where results[model][pool] = {oie_type: [candidates]}."""
    oie, entities, oie_types, type_texts = load_queries(article_id)
    results = {}
    for mname in EMBED_MODELS:
        qp, dp = embed_prefixes(mname)
        print(f"\n=== {mname}  query_prefix={qp!r} doc_prefix={dp!r} ===")
        m = get_model(mname)
        results[mname] = {pk: retrieve(m, pool, type_texts, oie_types, qp, dp)
                          for pk, pool in POOLS.items()}
    return oie, entities, oie_types, results

933 baseline classes | 933/933 enriched


## 2. Match the selected article

In [4]:
oie, entities, oie_types, results = match_article(ARTICLE_ID)
print("\nmodels:", list(results), "| pools:", list(POOLS))


=== BAAI/bge-small-en-v1.5  query_prefix='Represent this sentence for searching relevant passages: ' doc_prefix='' ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]


=== intfloat/e5-base-v2  query_prefix='query: ' doc_prefix='passage: ' ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]


models: ['BAAI/bge-small-en-v1.5', 'intfloat/e5-base-v2'] | pools: ['baseline', 'gemini']


## 3. Inspect results

In [5]:
# Which model/pool to inspect in the per-type views below.
import statistics
SHOW_MODEL = EMBED_MODELS[0]     # embedding model shown in the per-type detail cells
SHOW_POOL  = "gemini"            # pool to inspect: "baseline" | "gemini"

def margin(cands):
    return cands[0]["score"] - cands[1]["score"] if len(cands) > 1 else float("nan")

# Mean top1-top2 margin for every (embedding model x pool). Higher = more decisive matches.
print("mean top1-top2 margin  (higher = more decisive)\n")
print(f"{'embedding model':<26}" + "".join(f"{pk:>12}" for pk in POOLS))
for mname in EMBED_MODELS:
    row = f"{mname:<26}"
    for pk in POOLS:
        row += f"{statistics.mean(margin(results[mname][pk][t]) for t in oie_types):>12.3f}"
    print(row)

mean top1-top2 margin  (higher = more decisive)

embedding model               baseline      gemini
BAAI/bge-small-en-v1.5           0.045       0.069
intfloat/e5-base-v2              0.021       0.032


In [6]:
# Per-type top-1 for each embedding model (pool = SHOW_POOL). Format: label(score / margin).
print(f"top-1 schema match per OIE type   |   pool = {SHOW_POOL}\n")
print(f"{'OIE type':<34}" + "".join(f"{m.split('/')[-1]:<30}" for m in EMBED_MODELS))
for t in oie_types:
    row = f"{t:<34}"
    for mname in EMBED_MODELS:
        c = results[mname][SHOW_POOL][t]
        row += f"{c[0]['label']}({c[0]['score']:.2f}/m{margin(c):+.2f})".ljust(30)
    print(row)

top-1 schema match per OIE type   |   pool = gemini

OIE type                          bge-small-en-v1.5             e5-base-v2                    
Person                            Person(0.91/m+0.17)           Person(0.89/m+0.08)           
City                              City(0.91/m+0.20)             City(0.91/m+0.11)             
URL                               URL(0.94/m+0.26)              URL(0.92/m+0.12)              
Date                              Date(0.92/m+0.17)             Date(0.89/m+0.05)             
Venue                             Place(0.82/m+0.03)            Place(0.88/m+0.04)            
Time of Day                       Time(0.78/m+0.03)             Time(0.85/m+0.02)             
Vehicle                           Vehicle(0.94/m+0.08)          Vehicle(0.93/m+0.05)          
News Outlet                       NewsMediaOrganization(0.81/m+0.09)NewsMediaOrganization(0.84/m+0.02)
Quantity                          Quantity(0.92/m+0.07)         Quantity(0.89/m+0.02

In [7]:
# Do the two embedding models agree on top-1? (per pool) + full top-k for the chosen model/pool.
if len(EMBED_MODELS) >= 2:
    a, b = EMBED_MODELS[0], EMBED_MODELS[1]
    sa, sb = a.split('/')[-1], b.split('/')[-1]
    print(f"top-1 agreement  {sa}  vs  {sb}:")
    for pk in POOLS:
        agree = sum(results[a][pk][t][0]["label"] == results[b][pk][t][0]["label"] for t in oie_types)
        print(f"  {pk:<10} {agree}/{len(oie_types)} = {agree/len(oie_types):.0%}")

print(f"\nfull top-{TOP_K}   model={SHOW_MODEL.split('/')[-1]}  pool={SHOW_POOL}")
for t in oie_types:
    print(f"\n>> {t}")
    for c in results[SHOW_MODEL][SHOW_POOL][t]:
        print(f"    {c['score']:.3f}  schema:{c['label']:<24} ({c['path']})")

top-1 agreement  bge-small-en-v1.5  vs  e5-base-v2:
  baseline   22/42 = 52%
  gemini     29/42 = 69%

full top-5   model=bge-small-en-v1.5  pool=gemini

>> Person
    0.906  schema:Person                   (Thing)
    0.739  schema:Patient                  (Thing > Intangible > Audience > MedicalAudience)
    0.719  schema:Physician                (Thing > Organization > LocalBusiness > MedicalBusiness)
    0.716  schema:PeopleAudience           (Thing > Intangible > Audience)
    0.715  schema:Researcher               (Thing > Intangible > Audience)

>> City
    0.908  schema:City                     (Thing > Place > AdministrativeArea)
    0.711  schema:CityHall                 (Thing > Place > CivicStructure > GovernmentBuilding)
    0.689  schema:DefinedRegion            (Thing > Place)
    0.684  schema:Residence                (Thing > Place)
    0.678  schema:State                    (Thing > Place > AdministrativeArea)

>> URL
    0.940  schema:URL                      (Text)


## 4. Persist results

In [ ]:
# Persist the GEMINI-pool matches per embedding model into results/<id>/.
#   *_by_type.json : one entry per distinct OIE type (the actual matching unit)
#   *.json         : every entity with its type's match (handy for downstream joins)
def _slug(m): return m.split("/")[-1].replace(".", "_")

def persist(article_id, entities, results):
    out_dir = RESULTS_DIR / article_id
    out_dir.mkdir(parents=True, exist_ok=True)
    for mname in EMBED_MODELS:
        by_type = results[mname]["gemini"]
        (out_dir / f"schema_matching_{_slug(mname)}_by_type.json").write_text(
            json.dumps(by_type, indent=2, ensure_ascii=False))
        ent_rows = [{"entity": e["text"], "free_type": e["type"], "candidates": by_type[e["type"]]}
                    for e in entities]
        (out_dir / f"schema_matching_{_slug(mname)}.json").write_text(
            json.dumps(ent_rows, indent=2, ensure_ascii=False))
    print("wrote schema_matching_* ->", out_dir)

persist(ARTICLE_ID, entities, results)

## 5. Batch — match every article that has OIE entities

In [ ]:
# Runs matching for every results/<id>/oie_entities.json produced by notebook 2.
for fp in sorted(RESULTS_DIR.glob("*/oie_entities.json")):
    aid = fp.parent.name
    print("=" * 60, aid)
    _oie, _ents, _types, _res = match_article(aid)
    persist(aid, _ents, _res)
print("\ndone — all articles matched")

============================================================ art1

=== BAAI/bge-small-en-v1.5  query_prefix='Represent this sentence for searching relevant passages: ' doc_prefix='' ===


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]


=== intfloat/e5-base-v2  query_prefix='query: ' doc_prefix='passage: ' ===


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

wrote schema_matching_* -> results/art1
============================================================ art2

=== BAAI/bge-small-en-v1.5  query_prefix='Represent this sentence for searching relevant passages: ' doc_prefix='' ===


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

## 6. Type selection + within-type dedup  (Gemini, one call per article)

A **single Gemini call per article** processes all of its OIE types at once. For each type (group),
independently, it does two separate things:

1. **Type selection** (`chosen_type`) — pick the best schema.org candidate for the type, or *mint* a
   new type (label + one-line definition) only when none fits. Minting is the fallback, not the default.
2. **Within-type dedup** (`duplicate_clusters`) — cluster that type's mentions that refer to the same
   real-world entity, keeping the most complete surface form as canonical. **Conservative:** merge only
   clear same-entity cases; overlapping-but-distinct entities (`France`/`Météo-France`, `Paris`/`Paris region`)
   are never merged.

Dedup is **same-OIE-type only** — mentions are never merged across groups, and cross-type merging is left
for a later pass (`# TODO` in the code). If the single call fails to parse or omits some groups, those
groups fall back to their own per-type call (`fallback_per_type=True`), so one hiccup never sinks the article.
Each mention's evidence (`normalized_value`, `confidence`) is preserved onto its canonical node.
Output: a new `results/<id>/typed_deduped_entities.json` (never overwrites the `schema_matching_*` files).

In [39]:
# --- Config + Gemini client for the type-selection / dedup stage --------------------------
# This stage is LLM-based (not embeddings), so it needs its own Gemini client — same pattern as
# notebook 2. MATCH_MODEL selects which schema_matching_<model>_by_type.json supplies candidates.
%pip install -q google-genai

import os, time
from google import genai
from google.genai import types as genai_types

DEDUP_ARTICLE_ID = "art1"             # which article this stage runs on (defaults to the one set up top)
MATCH_MODEL   = "intfloat/e5-base-v2"     # which embedding model's *_by_type.json feeds candidates
GEMINI_MODEL  = "gemini-2.5-flash"

# Valid Schema.org class labels — used to sanity-check the PARENT superclass of any minted type.
_SCHEMA_LABELS = {c["label"] for c in json.loads((SCHEMA_DIR / "schema_classes.json").read_text())}

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env and restart the kernel."
client = genai.Client(api_key=GEMINI_API_KEY)

def gemini_text(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini in JSON mode; return the RAW response text (parsing is done separately so we
    can strip fences and handle errors per group)."""
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt, temperature=temperature,
            response_mime_type="application/json"),
    )
    return resp.text

def _parse_json_safe(text):
    """Strip optional ``` / ```json fences, then json.loads. Raises on malformed JSON (caller logs)."""
    s = (text or "").strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return json.loads(s)

print("type-dedup stage | article:", DEDUP_ARTICLE_ID, "| candidates from:", _slug(MATCH_MODEL),
      "| llm:", GEMINI_MODEL, "|", len(_SCHEMA_LABELS), "schema.org labels")

Note: you may need to restart the kernel to use updated packages.
type-dedup stage | article: art1 | candidates from: e5-base-v2 | llm: gemini-2.5-flash | 933 schema.org labels


In [40]:
# ==== EDIT ME =============================================================================
# The prompt for ONE call per article covering ALL OIE types. For EACH group independently:
# type selection FIRST, then within-group dedup, reported as SEPARATE fields per group.
TYPE_DEDUP_SYSTEM_PROMPT = """You are an expert knowledge-engineering assistant. You are given several groups of entity mentions from one document. Each group shares a single free-form OIE type. For each group independently, perform two tasks, in order, and report them as separate fields (chosen_type / duplicate_clusters) — do not blend the two tasks, and NEVER merge or compare mentions across different groups.

Each group provides:
- OIE_TYPE: the free-form type label these mentions were given, plus its definition.
- CANDIDATES: a ranked list of Schema.org types (label, path in the hierarchy, comment/definition) proposed as matches for OIE_TYPE. A candidate's `path` is its chain of ancestor classes (e.g. "Thing > Intangible > Enumeration"), so every class you might use as a parent is already listed here.
- MENTIONS: the verbatim surface strings that were all assigned OIE_TYPE.

TASK 1 — TYPE SELECTION  (output field: "chosen_type")
Choose the single best type for the group.
- Prefer a Schema.org candidate. Pick the one whose meaning (judged from its comment and path) best fits OIE_TYPE and the mentions. A candidate that is somewhat broader or narrower but still preserves what makes the type distinctive is preferable to a new type.
- Instance test — apply before accepting any candidate, judging by the MENTIONS, not by the OIE_TYPE definition alone: would each mention individually be a valid instance of that candidate — i.e. would it sit naturally alongside the concrete examples listed in the candidate's comment? If the mentions are open-ended categories, roles, or kinds rather than specific instances of the candidate, the candidate is too broad; reject it.
- Mint a new type when no candidate passes the instance test — i.e. no candidate genuinely fits, including when the best candidate is only "correct" by being so broad that it loses the type's distinguishing meaning. Do not settle for an over-broad candidate; instead mint a concise PascalCase label and a one-sentence definition that captures the type's specific meaning.
- When you mint, also assign a parent: the single closest existing Schema.org type the minted type should be a subclass of, so it slots into the hierarchy. Prefer a class already shown for this group — a candidate label, or any ancestor listed in a candidate's `path` (e.g. for path "Thing > Intangible > Enumeration", valid parents include Enumeration, Intangible, Thing). Use another real Schema.org class only if none of those fit, and fall back to "Thing" as a last resort. Parent is required for minted types and null otherwise.
- Report the chosen label, whether it is new, its definition and parent (both only when new; else null), and a one-line justification.

TASK 2 — WITHIN-GROUP DEDUPLICATION  (output field: "duplicate_clusters")
Cluster this group's mentions that refer to the same real-world entity. Only ever cluster mentions that belong to the same group.
- Be conservative. Merge two mentions ONLY when they clearly denote the same specific entity — e.g. a short form / abbreviation and its full form: "US" & "United States", "Swift" & "Taylor Swift", "the Garden" & "Madison Square Garden". When unsure, keep them separate. Under-merging is much better than over-merging.
- NEVER merge overlapping-but-distinct entities: "France" vs "Météo-France", "Paris" vs "Paris region", "New York" vs "New York City", a person vs an organization named after them.
- For each cluster pick the canonical mention: the most complete / most explicit surface form ("United States" over "US", "Taylor Swift" over "Swift"). The canonical and every alias must be exact strings copied from that group's mentions.
- Output only clusters of size >= 2. Do not list singletons.

Return ONLY valid JSON matching this schema — no prose, no code fences. Output one result object per input group, in the same order, each echoing its exact oie_type string. "parent" is a Schema.org label when "is_new" is true, else null:
{
  "results": [
    {
      "oie_type": "Country",
      "chosen_type": { "label": "Country", "is_new": false, "definition": null, "parent": null, "justification": "one short line" },
      "duplicate_clusters": [ { "canonical": "United States", "aliases": ["US"] } ]
    },
    {
      "oie_type": "Group of People",
      "chosen_type": { "label": "SomeMintedType", "is_new": true, "definition": "one sentence.", "parent": "Thing", "justification": "one short line" },
      "duplicate_clusters": []
    }
  ]
}
Include every group exactly once. If a group has nothing to merge, use "duplicate_clusters": []."""
# =========================================================================================
print(f"prompt loaded ({len(TYPE_DEDUP_SYSTEM_PROMPT)} chars)")

prompt loaded (4550 chars)


In [41]:
# Core: ONE batched Gemini call per ARTICLE (all OIE types at once) -> per-type selection +
# within-type dedup, merged into canonical entity nodes. If the single call fails to parse or omits
# some groups, those groups fall back to their own per-type call; metadata is preserved onto the canonical.
_META_KEYS = ("text", "normalized_value", "confidence")

def _load_group_inputs(article_id, match_model=MATCH_MODEL):
    """Group an article's entities by OIE type; attach each type's schema.org candidates + its def."""
    out_dir  = RESULTS_DIR / article_id
    oie      = json.loads((out_dir / "oie_entities.json").read_text())
    by_type  = json.loads((out_dir / f"schema_matching_{_slug(match_model)}_by_type.json").read_text())
    type_def = {t["type"]: t for t in oie.get("types", [])}
    groups   = {}                                   # oie_type -> [entity dict, ...] (order preserved)
    for e in oie["entities"]:
        groups.setdefault(e["type"], []).append(e)
    return groups, by_type, type_def

def _group_block(idx, oie_type, entities, candidates, type_def):
    """One GROUP block in the per-article message: OIE type + its candidates + its mentions.
    Similarity scores are deliberately NOT shown to the LLM — they are compressed into a narrow
    high band and read as false confidence; type selection is meaning-based (comment + path)."""
    lines = [f"GROUP {idx} —", f'OIE_TYPE: "{oie_type}"']
    if type_def.get("definition"):
        lines.append(f'OIE_TYPE_DEFINITION: {type_def["definition"]}')
    lines.append("CANDIDATES (Schema.org types, ranked):")
    if candidates:
        for i, c in enumerate(candidates, 1):
            path = c.get("path") or "(datatype)"
            lines.append(f'  {i}. {c["label"]} | path: {path}')
            lines.append(f'     {c.get("comment", "")}')
    else:
        lines.append("  (none)")
    lines.append("MENTIONS:")
    lines += [f'  - "{e["text"]}"' for e in entities]
    return "\n".join(lines)

def _article_user_message(groups, by_type, type_def):
    blocks = [_group_block(i, t, ents, by_type.get(t, []), type_def.get(t, {}))
              for i, (t, ents) in enumerate(groups.items(), 1)]
    return f"You are given {len(groups)} GROUPS from one document.\n\n" + "\n\n".join(blocks)

def _call_groups(groups, by_type, type_def):
    """One Gemini call for the given {oie_type: entities}; returns {oie_type: {chosen_type, clusters}}
    for whichever groups the model echoed back. Raises on a malformed/unparseable response."""
    raw     = gemini_text(TYPE_DEDUP_SYSTEM_PROMPT, _article_user_message(groups, by_type, type_def))
    parsed  = _parse_json_safe(raw)
    results = parsed.get("results", parsed) if isinstance(parsed, dict) else parsed
    out = {}
    for r in results or []:
        t = r.get("oie_type")
        if t in groups:
            out[t] = {"chosen_type": r.get("chosen_type") or {}, "clusters": r.get("duplicate_clusters") or []}
    return out

def _make_node(oie_type, chosen, canonical, alias_texts, by_text):
    is_new = bool(chosen.get("is_new"))
    members = [canonical] + alias_texts
    meta    = [by_text[t] for t in members if t in by_text]
    return {
        "canonical_text":    canonical,
        "aliases":           alias_texts,
        "oie_type":          oie_type,
        "chosen_type":       chosen.get("label"),
        "type_is_new":       is_new,
        "type_definition":   chosen.get("definition"),
        "type_parent":       chosen.get("parent") if is_new else None,   # schema.org superclass for minted types
        "type_justification": chosen.get("justification"),
        "mention_count":     len(meta),
        "mentions":          [{k: m.get(k) for k in _META_KEYS} for m in meta],
    }

def _apply_clusters(oie_type, entities, chosen, clusters):
    """Turn LLM clusters into canonical nodes; every mention lands in exactly one node.
    Conservative fallbacks: a cluster whose canonical isn't a real mention, or a mention already
    claimed by an earlier cluster, is dropped/skipped rather than force-merged (-> stays a singleton)."""
    by_text, used, nodes = {e["text"]: e for e in entities}, set(), []
    for cl in clusters or []:
        canonical = cl.get("canonical")
        if canonical not in by_text or canonical in used:
            continue                                              # hallucinated/duplicate -> skip
        aliases = [a for a in (cl.get("aliases") or [])
                   if a in by_text and a != canonical and a not in used]
        if not aliases:
            continue                                              # nothing left to merge
        for m in [canonical] + aliases:
            used.add(m)
        nodes.append(_make_node(oie_type, chosen, canonical, aliases, by_text))
    for e in entities:                                            # remaining mentions -> singletons
        if e["text"] not in used:
            used.add(e["text"])
            nodes.append(_make_node(oie_type, chosen, e["text"], [], by_text))
    return nodes

def type_and_dedup_article(article_id, match_model=MATCH_MODEL, write=True, verbose=True,
                           fallback_per_type=True):
    """ONE Gemini call for the whole article (all OIE types) -> type-selection + conservative
    within-type dedup. Groups the model omits (or the whole call, if it fails to parse) fall back to
    their own per-type call when fallback_per_type=True. Returns (canonical_nodes, report)."""
    groups, by_type, type_def = _load_group_inputs(article_id, match_model)
    report = {}
    try:
        report = _call_groups(groups, by_type, type_def)          # primary: single article-wide call
        if verbose:
            print(f"  single call: {len(report)}/{len(groups)} groups returned")
    except Exception as ex:
        print(f"  [WARN] {article_id}: article-wide call failed ({type(ex).__name__}: {ex})")

    missing = [t for t in groups if t not in report]
    if missing and fallback_per_type:                             # fill gaps one type at a time
        if verbose:
            print(f"  fallback: {len(missing)} group(s) via per-type calls")
        for t in missing:
            try:
                report.update(_call_groups({t: groups[t]}, by_type, type_def))
            except Exception as ex:
                print(f"  [WARN] {article_id} / {t!r}: {type(ex).__name__}: {ex} -> no type, no dedup")
    for t in groups:                                              # anything still missing -> singletons
        report.setdefault(t, {"chosen_type": {}, "clusters": []})

    # sanity: a minted type should hang off a REAL schema.org parent (warn, don't fail)
    for oie_type, r in report.items():
        ct = r.get("chosen_type") or {}
        if ct.get("is_new"):
            p = ct.get("parent")
            if not p or p not in _SCHEMA_LABELS:
                print(f"  [WARN] {article_id} / {oie_type!r}: minted {ct.get('label')!r} "
                      f"has parent {p!r} not in schema.org")

    nodes = []
    for oie_type, ents in groups.items():
        r = report[oie_type]
        nodes.extend(_apply_clusters(oie_type, ents, r["chosen_type"], r["clusters"]))
        if verbose:
            ct = r["chosen_type"] or {}
            tag = f"  <:{ct.get('parent')}" if ct.get("is_new") else ""
            print(f"  {oie_type:<32} -> {ct.get('label')!s:<24}{tag} "
                  f"({len(ents)} mentions, {len(r['clusters'])} clusters)")

    # TODO: cross-type dedup pass — the same real-world entity can surface under DIFFERENT OIE types
    # (e.g. "Taylor Swift" as `Person` and as `Person (Musician)`; an org and its named event).
    # That cross-type merge is intentionally NOT done here (this stage is same-OIE-type only) and
    # will be added as a separate pass (likely embedding-assisted).

    if write:
        out_fp = RESULTS_DIR / article_id / "typed_deduped_entities.json"   # new file; never overwrites schema_matching_*
        out_fp.write_text(json.dumps(
            {"article_id": article_id, "match_model": match_model, "entities": nodes},
            indent=2, ensure_ascii=False))
        if verbose:
            print(f"  wrote {len(nodes)} canonical entities -> {out_fp}")
    return nodes, report

### Test run — eyeball chosen type + clusters (no writes)

In [42]:
# Test run (no file writes): print per OIE type the chosen type + any duplicate clusters, so you
# can eyeball type selection and dedup quality before committing the write below.
TEST_ARTICLE = DEDUP_ARTICLE_ID     # the article configured for this stage (must have a *_by_type.json)

_nodes, _report = type_and_dedup_article(TEST_ARTICLE, write=False, verbose=True)

print(f"\n{'='*70}\n{TEST_ARTICLE}: per-OIE-type chosen type + duplicate clusters\n{'='*70}")
for oie_type, r in _report.items():
    ct = r["chosen_type"] or {}
    tag = f"NEW  subClassOf {ct.get('parent')}" if ct.get("is_new") else "schema.org"
    print(f"\n[{oie_type}]  ->  {ct.get('label')}  ({tag})")
    if ct.get("definition"):
        print(f"    def:    {ct['definition']}")
    if ct.get("justification"):
        print(f"    reason: {ct['justification']}")
    if r["clusters"]:
        for cl in r["clusters"]:
            print(f"    dup:    {cl.get('canonical')!r}  <=  {cl.get('aliases')}")
    else:
        print("    dup:    (none)")

  single call: 45/45 groups returned
  Country                          -> Country                  (6 mentions, 0 clusters)
  Quantity of Deaths               -> CountOfDeaths             <:Quantity (3 mentions, 0 clusters)
  Weather Phenomenon               -> Event                    (1 mentions, 0 clusters)
  Continent                        -> Continent                (1 mentions, 0 clusters)
  Date                             -> Date                     (3 mentions, 0 clusters)
  Month                            -> Month                     <:Intangible (1 mentions, 0 clusters)
  Temporal Expression              -> TemporalExpression        <:Intangible (8 mentions, 0 clusters)
  Person                           -> Person                   (4 mentions, 0 clusters)
  Government Official Title        -> GovernmentOfficialRole    <:Intangible (2 mentions, 0 clusters)
  Age                              -> Age                       <:Quantity (2 mentions, 0 clusters)
  Temperature    

### Write `typed_deduped_entities.json` for the configured article (`DEDUP_ARTICLE_ID`)

In [43]:
# Write results/<id>/typed_deduped_entities.json for the single configured article (DEDUP_ARTICLE_ID).
# New filename: never touches the schema_matching_* files. (To run everything, loop this over
# sorted(RESULTS_DIR.glob("*/oie_entities.json")).)
def _has_bytype(aid):
    return (RESULTS_DIR / aid / f"schema_matching_{_slug(MATCH_MODEL)}_by_type.json").exists()

aid = DEDUP_ARTICLE_ID
if not _has_bytype(aid):
    print(f"skip {aid}: no {_slug(MATCH_MODEL)}_by_type.json (run the matching batch first)")
else:
    print("=" * 60, aid)
    _nodes, _report = type_and_dedup_article(aid, write=True, verbose=False)
    _merged = sum(1 for n in _nodes if n["mention_count"] > 1)
    _minted = sum(1 for r in _report.values() if (r["chosen_type"] or {}).get("is_new"))
    print(f"  {len(_nodes)} canonical entities ({_merged} merged) | {_minted} minted types")
    print("done —", RESULTS_DIR / aid / "typed_deduped_entities.json")

============================================================ art1
  85 canonical entities (1 merged) | 23 minted types
done — results/art1/typed_deduped_entities.json


### Verify — read back `typed_deduped_entities.json` vs `oie_entities.json`

In [44]:
# Verify: read back every typed_deduped_entities.json and cross-check it against oie_entities.json.
# Invariants (same-OIE-type stage): the flattened (oie_type, mention_text) multiset must equal the
# original entities exactly — no mention lost, none duplicated across canonical nodes.
def verify_typed_deduped(article_id):
    out_dir = RESULTS_DIR / article_id
    oie   = json.loads((out_dir / "oie_entities.json").read_text())
    typed = json.loads((out_dir / "typed_deduped_entities.json").read_text())
    nodes = typed["entities"]

    orig = sorted((e["type"], e["text"]) for e in oie["entities"])
    got  = sorted((n["oie_type"], m["text"]) for n in nodes for m in n["mentions"])
    problems = []
    if orig != got:
        lost  = sorted(set(orig) - set(got))
        extra = sorted(set(got)  - set(orig))
        dupes = sorted(p for p in set(got) if got.count(p) > 1)
        if lost:  problems.append(f"{len(lost)} lost e.g. {lost[:3]}")
        if extra: problems.append(f"{len(extra)} unexpected e.g. {extra[:3]}")
        if dupes: problems.append(f"{len(dupes)} mention(s) in >1 node e.g. {dupes[:3]}")
    # per-node sanity: mention_count matches, canonical not in its own aliases
    for n in nodes:
        if n["mention_count"] != len(n["mentions"]):
            problems.append(f"count!=len(mentions) for {n['canonical_text']!r}")
        if n["canonical_text"] in n["aliases"]:
            problems.append(f"canonical in aliases for {n['canonical_text']!r}")

    merged = sum(1 for n in nodes if n["mention_count"] > 1)
    minted = sum(1 for n in nodes if n.get("type_is_new"))
    status = "OK" if not problems else "FAIL: " + "; ".join(problems)
    print(f"{article_id:<8} {len(oie['entities']):>4} mentions -> {len(nodes):>4} nodes "
          f"({merged} merged, {minted} minted-type nodes)   {status}")
    return not problems

print(f"{'article':<8} {'':>4} check\n" + "-" * 64)
_all_ok = True
for fp in sorted(RESULTS_DIR.glob("*/typed_deduped_entities.json")):
    _all_ok &= verify_typed_deduped(fp.parent.name)
print("-" * 64)
print("all articles consistent ✅" if _all_ok else "inconsistencies found ⚠️  (see FAIL rows above)")

article       check
----------------------------------------------------------------
art1       86 mentions ->   85 nodes (1 merged, 40 minted-type nodes)   OK
art2      113 mentions ->  105 nodes (6 merged, 25 minted-type nodes)   OK
art20      94 mentions ->   80 nodes (12 merged, 11 minted-type nodes)   OK
art22      43 mentions ->   40 nodes (3 merged, 8 minted-type nodes)   OK
----------------------------------------------------------------
all articles consistent ✅


## 7. Review export — one row per OIE type across all stages

A single human-readable file, `results/<id>/matching_review_<model>.json`, that stitches the whole
pipeline together per OIE type: the **query-side** text embedded for the type, the **document-side**
text of each candidate schema class (the two sides whose cosine similarity gave `score`), the
**mentions** the LLM saw, the ranked **candidates** it chose from, and the **final chosen type**
(picked or minted). Purely for inspection — nothing downstream reads it.

In [45]:
# Human-readable review export: one row per OIE type joining every stage of the pipeline —
#   - embedding_query_text : the string embedded on the QUERY side (OIE type + definition + synonyms)
#   - candidates[].embedding_doc_text : the string embedded on the DOCUMENT side for that schema class
#     (these two are exactly the pair whose cosine similarity produced `score`)
#   - mentions : the surface strings assigned this OIE type (what the LLM saw)
#   - final_chosen_type : the type the LLM picked/minted (uniform across the group)
# Writes results/<id>/matching_review_<model>.json ; a review artifact, never used downstream.
def export_matching_review(article_id, match_model=MATCH_MODEL, write=True, verbose=True):
    out_dir = RESULTS_DIR / article_id
    oie     = json.loads((out_dir / "oie_entities.json").read_text())
    by_type = json.loads((out_dir / f"schema_matching_{_slug(match_model)}_by_type.json").read_text())
    typed   = json.loads((out_dir / "typed_deduped_entities.json").read_text())

    doc_text = {c["label"]: c.get("embedding_text")                       # schema-side text per class
                for c in json.loads((SCHEMA_DIR / "schema_classes_enriched.json").read_text())}
    _o, _e, oie_types, type_texts = load_queries(article_id)              # OIE-side query text per type
    query_of = dict(zip(oie_types, type_texts))
    type_def = {t["type"]: t for t in oie.get("types", [])}

    mentions = {}                                                        # oie_type -> [mention text] (input order)
    for e in oie["entities"]:
        mentions.setdefault(e["type"], []).append(e["text"])
    final = {}                                                           # oie_type -> chosen type (uniform per group)
    for n in typed["entities"]:
        final.setdefault(n["oie_type"], {
            "label": n["chosen_type"], "is_new": n["type_is_new"], "parent": n.get("type_parent"),
            "definition": n.get("type_definition"), "justification": n.get("type_justification")})

    rows = [{
        "oie_type":             t,
        "oie_definition":       type_def.get(t, {}).get("definition"),
        "embedding_query_text": query_of.get(t),                         # <-- text 1 (query side)
        "final_chosen_type":    final.get(t),
        "mentions":             mentions.get(t, []),
        "candidates": [{
            "label":              c["label"],
            "score":              c.get("score"),
            "path":               c.get("path"),
            "embedding_doc_text": doc_text.get(c["label"]),              # <-- text 2 (document side)
            "comment":            c.get("comment"),
        } for c in by_type.get(t, [])],
    } for t in oie_types]

    review = {"article_id": article_id, "match_model": match_model, "types": rows}
    if write:
        fp = out_dir / f"matching_review_{_slug(match_model)}.json"
        fp.write_text(json.dumps(review, indent=2, ensure_ascii=False))
        if verbose:
            print(f"wrote {len(rows)} OIE types -> {fp}")
    return review

_review = export_matching_review(DEDUP_ARTICLE_ID)

wrote 45 OIE types -> results/art1/matching_review_e5-base-v2.json
